# 4. How Spark executes your code

The code of this module's lessons in [Learn Apache Spark by Doing](https://dataenglab.com/courses/learn-apache-spark-by-doing/), in lesson order, on Apache Spark 4.2.0 in local mode: one machine, no cluster.

Colab runs this notebook in your own Google account: sign in with Google to run the cells, and use *File → Save a copy in Drive* to keep your changes. Nothing runs on DataEngLab and nothing is installed on your computer.

Run the cells from top to bottom. Under a cell, the result the lesson shows is there to compare with yours.

Not available in Colab: the Spark UI (its port is not reachable from a notebook). Where a lesson sends you to it, run that part on your machine; here, `df.explain()` prints the plan.

In [ ]:
# Setup: Java 17+, PySpark 4.2.0 and the webshop data. About a minute the first time.
import os, re, subprocess, sys, urllib.request, zipfile

PYSPARK_VERSION = "4.2.0"
DATA_URL = 'https://github.com/dataeng-lab/learn-apache-spark-by-doing/raw/v2/datasets.zip'

if "google.colab" in sys.modules:
    version = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    major = re.search(r'version "(?:1\.)?(\d+)', version)
    if not major or int(major.group(1)) < 17:       # PySpark 4 does not start on Java 8 or 11
        subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless",
                       shell=True, check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"], check=True)

if not os.path.isdir("datasets"):
    if DATA_URL is None:
        raise RuntimeError("datasets/ is missing: upload the course's datasets folder in the Files panel")
    urllib.request.urlretrieve(DATA_URL, "datasets.zip")
    zipfile.ZipFile("datasets.zip").extractall(".")

print("datasets:", sorted(os.listdir("datasets")))

**Starting point.** The DataFrames and names the earlier modules built. Not part of the lesson text.

In [ ]:
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("webshop")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

from pyspark.sql.types import (
    DecimalType, IntegerType, StringType, StructField, StructType, TimestampType,
)

ORDERS_SCHEMA = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=True),
    StructField("order_ts", TimestampType(), nullable=True),
    StructField("status", StringType(), nullable=True),
    StructField("amount", DecimalType(10, 2), nullable=True),
    StructField("currency", StringType(), nullable=True),
    StructField("updated_at", TimestampType(), nullable=True),
])
orders_schema = ORDERS_SCHEMA   # some lessons use this name

orders = spark.read.csv("datasets/orders.csv", header=True, schema=ORDERS_SCHEMA)

## Transformations build a plan, actions run it

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/transformations-and-actions/)

In [ ]:
t0 = time.perf_counter()
df = (
    orders
    .filter(F.col("status") == "created")
    .withColumn("amount_eur", F.col("amount"))
    .groupBy("currency")
    .agg(F.sum("amount_eur").alias("revenue"))
)
t1 = time.perf_counter()          # four transformations built

rows = df.collect()               # one action
t2 = time.perf_counter()

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
building 4 transformations took 149.7 ms  (no data was read)
calling collect() took         1557.4 ms  (the work happened here)
result: [('EUR', '2526490.94')]
```

Not run here: `agg(...)` is a placeholder for any aggregation.

```python
orders_clean = orders.filter(F.col("status") == "created")

print(orders_clean.count())        # action 1: reads the file
print(orders_clean.agg(...).show())  # action 2: reads the file AGAIN
```

## Lazy evaluation: what is deferred and what is not

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/lazy-evaluation-what-is-deferred/)

## Narrow and wide: does the data have to move?

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/narrow-and-wide-transformations/)

Not run here: `spark.read.csv(...)` is a placeholder; `orders` is already loaded.

```python
orders = spark.read.csv(...).repartition(4)
narrow = orders.filter(F.col("amount") > 100).withColumn("big", F.lit(True))
```

In [ ]:
wide = orders.groupBy("status").agg(F.count("*").alias("n"))

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
spark.sql.shuffle.partitions = 200
partitions out: 1
```

## Inside a shuffle

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/inside-a-shuffle/)

## Reading an execution plan

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/reading-execution-plans/)